# 🏋️ FormFix — GPU-Accelerated Multi-Class Training (All 26 Exercises)

This Google Colab notebook trains **three tuned deep-learning architectures** on CUDA GPU for 26 gym exercises:

| Model | Tuned Architecture & Hyperparameters | Key Advantages |
|:------|:-------------------------------------|:---------------|
| **BiLSTM** | 3 BiLSTM layers (256→128→64), Dual Avg+Max Pooling, 256→128 GELU head, L2 reg (1e-4), Dropout (0.35→0.3→0.2) | Captures past & future temporal dynamics + peak flexion frames |
| **LSTM** | 3 LSTM layers (256→128→64), 128→64 GELU head, LayerNorm, L2 reg (1e-4), Graduated Dropout (0.3→0.25→0.2) | Fast sequential baseline with smooth temporal gradients |
| **Transformer** | 4 Blocks, 8 Attention Heads (dim=128, ff=256), Pre-LN, Positional Encoding, Dual Pooling | Global self-attention across 30-frame exercise cycles |

### 📋 All 26 Supported Exercises:
1. `barbell_bench_press` &nbsp; 2. `barbell_curl` &nbsp; 3. `barbell_row` &nbsp; 4. `barbell_squat` &nbsp; 5. `bicep_curl`  
6. `cable_fly` &nbsp; 7. `deadlift` &nbsp; 8. `decline_bench_press` &nbsp; 9. `dips` &nbsp; 10. `flat_bench_press`  
11. `hammer_curl` &nbsp; 12. `hip_thrust` &nbsp; 13. `incline_bench_press` &nbsp; 14. `lat_pulldown` &nbsp; 15. `lateral_raise`  
16. `leg_extension` &nbsp; 17. `leg_raises` &nbsp; 18. `military_press` &nbsp; 19. `overhead_press` &nbsp; 20. `plank`  
21. `pull_up` &nbsp; 22. `push_up` &nbsp; 23. `romanian_deadlift` &nbsp; 24. `russian_twist` &nbsp; 25. `squat` &nbsp; 26. `tricep_pushdown`

---
### 🚀 How to Run with Automatic Download:
1. **Enable GPU**: Click **Runtime** → **Change runtime type** → Hardware accelerator: **T4 GPU** (or A100).
2. **Mount Google Drive** or upload dataset.
3. Click **Runtime** → **Run all**.
4. When training finishes, the notebook **automatically bundles all models into a single zip and downloads it to your browser**!


## 1. Setup & CUDA GPU Check

In [ ]:
import tensorflow as tf
import numpy as np
import os, sys, math, time, json, zipfile
import matplotlib.pyplot as plt

print(f"Python: {sys.version.split()[0]}")
print(f"TensorFlow version: {tf.__version__}")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ CUDA GPU Available: {gpus[0].name}")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("⚠️ WARNING: No GPU detected! Please go to: Runtime -> Change runtime type -> Select T4 GPU.")

!pip install -q scikit-learn "mediapipe<0.10.20" opencv-python
print("✅ Environment ready!")


## 2. Load Dataset (Choose Method A or B)

- **Method A (Recommended - Fastest)**: Upload preprocessed `exercise_sequences.npz` (via Drive or Colab file upload).
- **Method B (Extract from Videos)**: Upload `raw_videos.zip` and let Colab extract MediaPipe landmarks with adaptive balancing!


In [ ]:
from google.colab import drive, files

# ── Mount Google Drive ──
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/FormFix'
os.makedirs(DRIVE_DIR, exist_ok=True)
DRIVE_MODELS_DIR = os.path.join(DRIVE_DIR, 'trained_models')
os.makedirs(DRIVE_MODELS_DIR, exist_ok=True)

DRIVE_NPZ = os.path.join(DRIVE_DIR, 'exercise_sequences.npz')
LOCAL_NPZ = '/content/exercise_sequences.npz'

if os.path.exists(DRIVE_NPZ):
    import shutil
    shutil.copy2(DRIVE_NPZ, LOCAL_NPZ)
    print(f"✅ Found and copied exercise_sequences.npz from Drive ({os.path.getsize(LOCAL_NPZ) / 1e6:.1f} MB)")
elif os.path.exists(LOCAL_NPZ):
    print(f"✅ Using existing /content/exercise_sequences.npz")
else:
    print("ℹ️ No dataset found in Drive. You can upload exercise_sequences.npz or raw_videos.zip below:")
    uploaded = files.upload()
    for fname in uploaded:
        if fname.endswith('.npz'):
            import shutil
            shutil.copy2(fname, LOCAL_NPZ)
            print(f"✅ Loaded {fname} as training dataset!")


### (Optional) Method B: Extract Landmarks directly from `raw_videos.zip`
Run this cell **ONLY** if you uploaded `raw_videos.zip` instead of `exercise_sequences.npz`. It automatically applies **adaptive stride balancing** across all 26 exercises.


In [ ]:
# Self-healing install & self-contained setup
!pip install -q "mediapipe<0.10.20" opencv-python scikit-learn

import os, sys, math, shutil, zipfile
from collections import defaultdict
from pathlib import Path
import cv2
import mediapipe as mp

# Ensure Google Drive is mounted if possible
if not os.path.exists('/content/drive/MyDrive'):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        print("Drive mount skipped or already mounted.")

# Define paths safely
DRIVE_DIR = '/content/drive/MyDrive/FormFix'
os.makedirs(DRIVE_DIR, exist_ok=True)
LOCAL_NPZ = '/content/exercise_sequences.npz'
ZIP_PATH = '/content/raw_videos.zip'
DRIVE_ZIP = os.path.join(DRIVE_DIR, 'raw_videos.zip')

if os.path.exists(DRIVE_ZIP) and not os.path.exists(ZIP_PATH):
    print(f"Copying {DRIVE_ZIP} to local workspace...")
    shutil.copy2(DRIVE_ZIP, ZIP_PATH)

if not os.path.exists(ZIP_PATH) and not os.path.exists(LOCAL_NPZ):
    print("⚠️ raw_videos.zip not found in Drive or /content.")
    print("Please upload raw_videos.zip now:")
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded:
        if fname.endswith('.zip'):
            shutil.copy2(fname, ZIP_PATH)
            break

if os.path.exists(ZIP_PATH) and not os.path.exists(LOCAL_NPZ):
    print(f"📦 Extracting {ZIP_PATH}...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall('/content/raw_dataset')

    # Find raw_videos folder
    base_dir = Path('/content/raw_dataset')
    if (base_dir / 'raw_videos').exists():
        raw_videos_dir = base_dir / 'raw_videos'
    else:
        raw_videos_dir = base_dir

    def normalize_landmarks(landmarks):
        if len(landmarks) < 33:
            row = []
            for lm in landmarks: row.extend([lm.x, lm.y, lm.z, lm.visibility])
            return row
        l_hip, r_hip = landmarks[23], landmarks[24]
        hx, hy, hz = (l_hip.x + r_hip.x)/2, (l_hip.y + r_hip.y)/2, (l_hip.z + r_hip.z)/2
        l_sh, r_sh = landmarks[11], landmarks[12]
        sx, sy, sz = (l_sh.x + r_sh.x)/2, (l_sh.y + r_sh.y)/2, (l_sh.z + r_sh.z)/2
        scale = max(math.sqrt((sx-hx)**2 + (sy-hy)**2 + (sz-hz)**2), 1e-4)
        row = []
        for lm in landmarks:
            row.extend([(lm.x - hx)/scale, (lm.y - hy)/scale, (lm.z - hz)/scale, lm.visibility])
        return row

    mp_pose = mp.solutions.pose
    exercise_dirs = sorted([d for d in raw_videos_dir.iterdir() if d.is_dir()])
    print(f"Found {len(exercise_dirs)} candidate exercise folders.")

    class_sequences = defaultdict(list)
    SEQ_LEN = 30
    MAX_PER_CLASS = 450

    with mp_pose.Pose(static_image_mode=False, model_complexity=1) as pose:
        for ex_dir in exercise_dirs:
            vids = sorted(list(ex_dir.glob('*.mp4')) + list(ex_dir.glob('*.mov')) + list(ex_dir.glob('*.avi')))
            if not vids: continue
            label = ex_dir.name
            # Adaptive stride balancing:
            stride = 5 if len(vids) <= 12 else (8 if len(vids) <= 22 else 18)
            print(f"Extracting {label} ({len(vids)} vids, stride={stride})...")
            seqs = []
            for v_path in vids:
                cap = cv2.VideoCapture(str(v_path))
                frames = []
                while cap.isOpened():
                    ret, frame = cap.read()
                    if not ret: break
                    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    res = pose.process(rgb)
                    if res.pose_landmarks:
                        frames.append(normalize_landmarks(res.pose_landmarks.landmark))
                cap.release()
                for i in range(0, max(len(frames) - SEQ_LEN + 1, 0), stride):
                    w = frames[i : i + SEQ_LEN]
                    if len(w) == SEQ_LEN: seqs.append(w)
            if MAX_PER_CLASS > 0 and len(seqs) > MAX_PER_CLASS:
                idx = np.linspace(0, len(seqs)-1, MAX_PER_CLASS, dtype=int)
                seqs = [seqs[i] for i in idx]
            class_sequences[label] = seqs
            print(f"  -> {label}: {len(seqs)} sequences")

    label_names = sorted(class_sequences.keys())
    l2i = {l: i for i, l in enumerate(label_names)}
    X_all, y_all = [], []
    for l in label_names:
        X_all.extend(class_sequences[l])
        y_all.extend([l2i[l]] * len(class_sequences[l]))
    X_all, y_all = np.asarray(X_all, dtype=np.float32), np.asarray(y_all, dtype=np.int64)
    np.savez_compressed(LOCAL_NPZ, X=X_all, y=y_all, label_names=np.asarray(label_names))
    print(f"✅ Created {LOCAL_NPZ}: {len(X_all)} sequences, {len(label_names)} classes!")
else:
    print("Skipping video extraction; using existing dataset.")


## 3. Inspect Dataset & Class Balance (All 26 Classes)

In [ ]:
data = np.load(LOCAL_NPZ, allow_pickle=True)
X = data['X'].astype(np.float32)
y = data['y'].astype(np.int64)
label_names = [str(l) for l in data['label_names']]
num_classes = len(label_names)

print("=" * 65)
print(f"  FormFix Multi-Class Dataset Summary")
print("=" * 65)
print(f"  Total Sequences:     {X.shape[0]:,}")
print(f"  Sequence Length:     {X.shape[1]} frames")
print(f"  Features per Frame:  {X.shape[2]} (33 landmarks × 4 coords)")
print(f"  Active Classes:      {num_classes}")
print("=" * 65)

counts = np.bincount(y, minlength=num_classes)
max_c = max(counts.max(), 1)
print(f"\n{'Class Name':<28s} {'Sequences':>10s}  {'Distribution'}")
print("-" * 65)
for idx, name in enumerate(label_names):
    bar = '█' * int(counts[idx] / max_c * 25)
    print(f"{name:<28s} {counts[idx]:>10d}  {bar}")

imbalance = counts.max() / max(counts.min(), 1)
print("-" * 65)
print(f"Imbalance Ratio: {imbalance:.2f}x (Min: {counts.min()}, Max: {counts.max()})")

# Save label names mapping for export
with open('/content/label_names.json', 'w') as f:
    json.dump(label_names, f, indent=2)


## 4. Train/Test Split & Balanced Weighting

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.utils import to_categorical

# Stratified 80/20 train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Balanced class weights to eliminate loss bias against smaller classes
class_weights = compute_class_weight(
    'balanced', classes=np.arange(num_classes), y=y_train
)
class_weight_map = {i: float(w) for i, w in enumerate(class_weights)}

# One-hot encoded labels for Label-Smoothed Categorical Crossentropy
y_train_cat = to_categorical(y_train, num_classes)
y_test_cat = to_categorical(y_test, num_classes)

input_shape = (X.shape[1], X.shape[2])

print(f"✅ Training Sequences:   {X_train.shape[0]:,}")
print(f"✅ Test Sequences:       {X_test.shape[0]:,}")
print(f"✅ Input Shape:          {input_shape}")
print(f"✅ Output Classes:        {num_classes}")


## 5. Tuned Training Utilities

- **Linear Warmup + Cosine Annealing Decay**: Prevents early divergence on deep networks and converges to flat, generalizable minima.
- **Label Smoothing (0.1)**: Softens hard 0/1 targets to prevent overconfidence across 26 classes.


In [ ]:
class WarmupCosineDecay(tf.keras.optimizers.schedules.LearningRateSchedule):
    """Linear warmup followed by cosine annealing schedule."""
    def __init__(self, base_lr, warmup_steps, total_steps, min_lr=1e-6):
        super().__init__()
        self.base_lr = float(base_lr)
        self.warmup_steps = int(warmup_steps)
        self.total_steps = int(total_steps)
        self.min_lr = float(min_lr)

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup_pct = tf.minimum(
            step / tf.maximum(tf.cast(self.warmup_steps, tf.float32), 1.0), 1.0
        )
        decay_steps = tf.cast(self.total_steps - self.warmup_steps, tf.float32)
        decay_pct = tf.minimum(
            (step - tf.cast(self.warmup_steps, tf.float32)) / tf.maximum(decay_steps, 1.0),
            1.0
        )
        decay_pct = tf.maximum(decay_pct, 0.0)
        cosine = 0.5 * (1.0 + tf.cos(np.float32(math.pi) * decay_pct))
        lr = self.min_lr + (self.base_lr - self.min_lr) * cosine
        return lr * warmup_pct

    def get_config(self):
        return {
            "base_lr": self.base_lr,
            "warmup_steps": self.warmup_steps,
            "total_steps": self.total_steps,
            "min_lr": self.min_lr,
        }


def compile_tuned_model(model, base_lr, warmup_epochs, total_epochs, batch_size, num_train, label_smoothing=0.1):
    steps_per_epoch = max(num_train // batch_size, 1)
    warmup_steps = warmup_epochs * steps_per_epoch
    total_steps = total_epochs * steps_per_epoch

    lr_schedule = WarmupCosineDecay(base_lr, warmup_steps, total_steps)
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule, clipnorm=1.0)
    loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=label_smoothing)
    model.compile(optimizer=optimizer, loss=loss, metrics=['accuracy'])
    return model


def get_callbacks(model_name, patience=12):
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy', patience=patience, restore_best_weights=True, verbose=1
        ),
        tf.keras.callbacks.ModelCheckpoint(
            f'/content/{model_name}_best.keras',
            monitor='val_accuracy', save_best_only=True, verbose=1
        ),
    ]

print("✅ Training schedules and callbacks configured.")


## 6. Tuned Model Architectures

### 6.1 Tuned 3-Layer LSTM
- **3 LSTM Layers**: 256 → 128 → 64 units
- **Graduated Dropout**: 0.30 → 0.25 → 0.20
- **LayerNormalization** after each recurrent layer
- **L2 Regularization** (`1e-4`) on kernels
- **2-Stage GELU head**: 128 → 64 → 26


In [ ]:
from tensorflow.keras.layers import (
    Dense, Dropout, LSTM, Bidirectional, LayerNormalization,
    Input, InputLayer, GlobalAveragePooling1D, GlobalMaxPooling1D,
    Concatenate, MultiHeadAttention
)
from tensorflow.keras.models import Sequential, Model

def build_lstm(input_shape, num_classes):
    reg = tf.keras.regularizers.l2(1e-4)
    model = Sequential([
        InputLayer(input_shape=input_shape),
        LSTM(256, return_sequences=True, kernel_regularizer=reg),
        LayerNormalization(),
        Dropout(0.30),
        LSTM(128, return_sequences=True, kernel_regularizer=reg),
        LayerNormalization(),
        Dropout(0.25),
        LSTM(64, return_sequences=False),
        LayerNormalization(),
        Dropout(0.20),
        Dense(128, activation="gelu"),
        LayerNormalization(),
        Dropout(0.15),
        Dense(64, activation="gelu"),
        Dropout(0.10),
        Dense(num_classes, activation="softmax"),
    ], name="ExerciseLSTM")
    return model

lstm_model = build_lstm(input_shape, num_classes)
print(f"✅ LSTM Built — Total Parameters: {lstm_model.count_params():,}")


### 6.2 Tuned 3-Layer BiLSTM with Dual Average + Max Pooling
- **3 Bidirectional LSTMs**: 256 → 128 → 64 units per direction
- **Dual Pooling**: `GlobalAveragePooling1D` (tempo/motion context) + `GlobalMaxPooling1D` (peak frame contraction)
- **Classification Head**: 256 (GELU) → 128 (GELU) → 26 (Softmax)


In [ ]:
def build_bilstm(input_shape, num_classes):
    reg = tf.keras.regularizers.l2(1e-4)
    inputs = Input(shape=input_shape)

    x = Bidirectional(LSTM(256, return_sequences=True, kernel_regularizer=reg))(inputs)
    x = LayerNormalization()(x)
    x = Dropout(0.35)(x)

    x = Bidirectional(LSTM(128, return_sequences=True, kernel_regularizer=reg))(x)
    x = LayerNormalization()(x)
    x = Dropout(0.30)(x)

    x = Bidirectional(LSTM(64, return_sequences=True))(x)
    x = LayerNormalization()(x)
    x = Dropout(0.20)(x)

    # Dual temporal pooling: captures overall movement cycle + peak extension/contraction frame
    avg_pool = GlobalAveragePooling1D()(x)
    max_pool = GlobalMaxPooling1D()(x)
    pooled = Concatenate()([avg_pool, max_pool])

    dense1 = Dense(256, activation="gelu")(pooled)
    norm1 = LayerNormalization()(dense1)
    drop1 = Dropout(0.20)(norm1)

    dense2 = Dense(128, activation="gelu")(drop1)
    drop2 = Dropout(0.15)(dense2)

    outputs = Dense(num_classes, activation="softmax")(drop2)
    return Model(inputs, outputs, name="PostureBiLSTM")

bilstm_model = build_bilstm(input_shape, num_classes)
print(f"✅ BiLSTM Built — Total Parameters: {bilstm_model.count_params():,}")


### 6.3 Tuned 4-Block, 8-Head Transformer
- **Embedding Projection**: 132 landmark features → 128-dim dense embedding
- **Sinusoidal Positional Encoding**: encodes 30-frame temporal order
- **4 Pre-LN Transformer Blocks**: 8 attention heads with `key_dim = 16`, `ff_dim = 256`, `GELU`
- **Dual Pooling**: Average + Max temporal pooling


In [ ]:
class PositionalEncoding(tf.keras.layers.Layer):
    def __init__(self, sequence_length, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.sequence_length = sequence_length
        self.embed_dim = embed_dim
        self.pos_encoding = self._positional_encoding(sequence_length, embed_dim)

    def _positional_encoding(self, seq_len, d_model):
        pos = tf.range(seq_len, dtype=tf.float32)[:, tf.newaxis]
        i = tf.range(d_model, dtype=tf.float32)[tf.newaxis, :]
        angle_rates = 1.0 / tf.pow(10000.0, (2.0 * (i // 2.0)) / tf.cast(d_model, tf.float32))
        angle_rads = pos * angle_rates
        sines = tf.math.sin(angle_rads[:, 0::2])
        cosines = tf.math.cos(angle_rads[:, 1::2])
        pos_encoding = tf.concat([sines, cosines], axis=-1)
        return tf.cast(pos_encoding[tf.newaxis, ...], dtype=tf.float32)

    def call(self, inputs):
        seq_len = tf.shape(inputs)[1]
        return inputs + self.pos_encoding[:, :seq_len, :]

    def get_config(self):
        config = super().get_config()
        config.update({"sequence_length": self.sequence_length, "embed_dim": self.embed_dim})
        return config


class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim=128, num_heads=8, ff_dim=256, dropout_rate=0.20, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.ff_dim = ff_dim
        self.dropout_rate = dropout_rate

        self.att = MultiHeadAttention(
            num_heads=num_heads, key_dim=max(embed_dim // num_heads, 16), dropout=dropout_rate
        )
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="gelu"),
            Dropout(dropout_rate),
            Dense(embed_dim),
        ])
        self.norm1 = LayerNormalization(epsilon=1e-6)
        self.norm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(dropout_rate)
        self.dropout2 = Dropout(dropout_rate)

    def call(self, inputs, training=False):
        # Pre-LN
        x_norm1 = self.norm1(inputs)
        attn = self.att(x_norm1, x_norm1, training=training)
        attn = self.dropout1(attn, training=training)
        out1 = inputs + attn

        x_norm2 = self.norm2(out1)
        ffn = self.ffn(x_norm2, training=training)
        ffn = self.dropout2(ffn, training=training)
        return out1 + ffn

    def get_config(self):
        config = super().get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "num_heads": self.num_heads,
            "ff_dim": self.ff_dim,
            "dropout_rate": self.dropout_rate,
        })
        return config


def build_transformer(input_shape, num_classes, embed_dim=128, num_heads=8, ff_dim=256, num_blocks=4, dropout_rate=0.20):
    inputs = Input(shape=input_shape)
    x = Dense(embed_dim)(inputs)
    x = PositionalEncoding(sequence_length=input_shape[0], embed_dim=embed_dim)(x)
    x = Dropout(0.15)(x)

    for _ in range(num_blocks):
        x = TransformerBlock(embed_dim=embed_dim, num_heads=num_heads, ff_dim=ff_dim, dropout_rate=dropout_rate)(x)

    avg_pool = GlobalAveragePooling1D()(x)
    max_pool = GlobalMaxPooling1D()(x)
    pooled = Concatenate()([avg_pool, max_pool])

    dense1 = Dense(128, activation="gelu")(pooled)
    norm1 = LayerNormalization()(dense1)
    drop1 = Dropout(0.15)(norm1)

    dense2 = Dense(64, activation="gelu")(drop1)
    drop2 = Dropout(0.10)(dense2)

    outputs = Dense(num_classes, activation="softmax")(drop2)
    return Model(inputs, outputs, name="PostureTransformer")

transformer_model = build_transformer(input_shape, num_classes)
print(f"✅ Transformer Built — Total Parameters: {transformer_model.count_params():,}")


## 7. Model Training on GPU

### 7.1 Train Tuned LSTM

In [ ]:
LSTM_EPOCHS = 50
LSTM_BATCH = 32
LSTM_LR = 5e-4
LSTM_WARMUP = 3

lstm_model = build_lstm(input_shape, num_classes)
lstm_model = compile_tuned_model(
    lstm_model, base_lr=LSTM_LR, warmup_epochs=LSTM_WARMUP,
    total_epochs=LSTM_EPOCHS, batch_size=LSTM_BATCH,
    num_train=len(X_train), label_smoothing=0.1
)

print(f"🚀 Training LSTM on GPU ({lstm_model.count_params():,} params)...")
t0 = time.time()
lstm_history = lstm_model.fit(
    X_train, y_train_cat,
    epochs=LSTM_EPOCHS,
    batch_size=LSTM_BATCH,
    validation_data=(X_test, y_test_cat),
    callbacks=get_callbacks('exercise_lstm', patience=12),
    class_weight=class_weight_map,
    verbose=1,
)
lstm_time = time.time() - t0
print(f"\n✅ LSTM Finished in {lstm_time/60:.1f} min — Best Val Acc: {max(lstm_history.history['val_accuracy']):.4f}")


### 7.2 Train Tuned BiLSTM

In [ ]:
BILSTM_EPOCHS = 60
BILSTM_BATCH = 32
BILSTM_LR = 5e-4
BILSTM_WARMUP = 3

bilstm_model = build_bilstm(input_shape, num_classes)
bilstm_model = compile_tuned_model(
    bilstm_model, base_lr=BILSTM_LR, warmup_epochs=BILSTM_WARMUP,
    total_epochs=BILSTM_EPOCHS, batch_size=BILSTM_BATCH,
    num_train=len(X_train), label_smoothing=0.1
)

print(f"🚀 Training BiLSTM on GPU ({bilstm_model.count_params():,} params)...")
t0 = time.time()
bilstm_history = bilstm_model.fit(
    X_train, y_train_cat,
    epochs=BILSTM_EPOCHS,
    batch_size=BILSTM_BATCH,
    validation_data=(X_test, y_test_cat),
    callbacks=get_callbacks('exercise_bilstm', patience=12),
    class_weight=class_weight_map,
    verbose=1,
)
bilstm_time = time.time() - t0
print(f"\n✅ BiLSTM Finished in {bilstm_time/60:.1f} min — Best Val Acc: {max(bilstm_history.history['val_accuracy']):.4f}")


### 7.3 Train Tuned Transformer

In [ ]:
TRANSFORMER_EPOCHS = 70
TRANSFORMER_BATCH = 64
TRANSFORMER_LR = 3e-4
TRANSFORMER_WARMUP = 5

transformer_model = build_transformer(input_shape, num_classes)
transformer_model = compile_tuned_model(
    transformer_model, base_lr=TRANSFORMER_LR, warmup_epochs=TRANSFORMER_WARMUP,
    total_epochs=TRANSFORMER_EPOCHS, batch_size=TRANSFORMER_BATCH,
    num_train=len(X_train), label_smoothing=0.1
)

print(f"🚀 Training Transformer on GPU ({transformer_model.count_params():,} params)...")
t0 = time.time()
transformer_history = transformer_model.fit(
    X_train, y_train_cat,
    epochs=TRANSFORMER_EPOCHS,
    batch_size=TRANSFORMER_BATCH,
    validation_data=(X_test, y_test_cat),
    callbacks=get_callbacks('posture_transformer', patience=15),
    class_weight=class_weight_map,
    verbose=1,
)
transformer_time = time.time() - t0
print(f"\n✅ Transformer Finished in {transformer_time/60:.1f} min — Best Val Acc: {max(transformer_history.history['val_accuracy']):.4f}")


## 8. Evaluation, Confusion Matrix & Performance Metrics (All 26 Classes)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score

models_eval = {
    'LSTM': tf.keras.models.load_model('/content/exercise_lstm_best.keras'),
    'BiLSTM': tf.keras.models.load_model('/content/exercise_bilstm_best.keras'),
    'Transformer': tf.keras.models.load_model(
        '/content/posture_transformer_best.keras',
        custom_objects={'TransformerBlock': TransformerBlock, 'PositionalEncoding': PositionalEncoding}
    ),
}

results = {}
print("=" * 70)
print(f"  {'Model':<15s} {'Accuracy':>10s} {'F1 (Macro)':>15s} {'F1 (Weighted)':>15s}")
print("=" * 70)

for name, m in models_eval.items():
    preds = m.predict(X_test, verbose=0)
    pred_idx = np.argmax(preds, axis=1)
    acc = float(np.mean(pred_idx == y_test))
    f1_mac = f1_score(y_test, pred_idx, average='macro')
    f1_wt = f1_score(y_test, pred_idx, average='weighted')
    results[name] = {'acc': acc, 'f1_macro': f1_mac, 'f1_weighted': f1_wt, 'preds': pred_idx}
    print(f"  {name:<15s} {acc:>10.4f} {f1_mac:>15.4f} {f1_wt:>15.4f}")

print("=" * 70)
best_m = max(results, key=lambda k: results[k]['acc'])
print(f"\n🏆 Top Performing Model: {best_m} ({results[best_m]['acc']:.4f} accuracy)")


In [ ]:
# Classification Report for Best Model across 26 classes
print("=" * 75)
print(f"  Detailed Classification Report: {best_m} ({num_classes} Classes)")
print("=" * 75)
print(classification_report(
    y_test, results[best_m]['preds'],
    target_names=label_names, zero_division=0
))


In [ ]:
# 26-Class Confusion Matrix Visualizer
fig, ax = plt.subplots(figsize=(14, 12))
cm = confusion_matrix(y_test, results[best_m]['preds'])
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
ax.figure.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax.set_title(f'Confusion Matrix — {best_m} (Accuracy: {results[best_m]["acc"]:.4f})', fontsize=14, pad=15)
ax.set_xlabel('Predicted Label', fontsize=11)
ax.set_ylabel('True Label', fontsize=11)
ax.set_xticks(range(num_classes))
ax.set_yticks(range(num_classes))
ax.set_xticklabels(label_names, rotation=60, ha='right', fontsize=9)
ax.set_yticklabels(label_names, fontsize=9)

for i in range(num_classes):
    for j in range(num_classes):
        val = cm[i, j]
        if val > 0:
            color = 'white' if val > cm.max() / 2 else 'black'
            ax.text(j, i, str(val), ha='center', va='center', color=color, fontsize=8)

plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=180)
plt.show()
print("✅ Saved /content/confusion_matrix.png")


In [ ]:
# Training and Validation Accuracy/Loss Curves
fig, axes = plt.subplots(1, 3, figsize=(21, 5))
histories = {'LSTM': lstm_history, 'BiLSTM': bilstm_history, 'Transformer': transformer_history}

for idx, (m_name, hist) in enumerate(histories.items()):
    ax = axes[idx]
    ax.plot(hist.history['accuracy'], label='Train Acc', linewidth=2)
    ax.plot(hist.history['val_accuracy'], label='Val Acc', linewidth=2)
    best_ep = np.argmax(hist.history['val_accuracy'])
    best_val = hist.history['val_accuracy'][best_ep]
    ax.axvline(best_ep, color='red', linestyle='--', alpha=0.6, label=f'Best ({best_val:.3f})')
    ax.set_title(f'{m_name}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.legend()
    ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=180)
plt.show()
print("✅ Saved /content/training_curves.png")


## 9. Automatic Zip Packaging & Download

This cell executes **automatically** at the end of training:
1. Gathers all 3 best `.keras` model checkpoints.
2. Includes `label_names.json` (the 26-class index mapping).
3. Includes `confusion_matrix.png` & `training_curves.png`.
4. Saves them to Google Drive: `MyDrive/FormFix/trained_models/`.
5. **Automatically triggers browser download** of `formfix_models_26_classes.zip` directly to your local computer!


In [ ]:
import shutil, zipfile
from google.colab import files

artifacts = {
    'exercise_lstm.keras': '/content/exercise_lstm_best.keras',
    'exercise_bilstm.keras': '/content/exercise_bilstm_best.keras',
    'posture_transformer.keras': '/content/posture_transformer_best.keras',
    'label_names.json': '/content/label_names.json',
    'confusion_matrix.png': '/content/confusion_matrix.png',
    'training_curves.png': '/content/training_curves.png',
}

# 1. Back up to Google Drive
print(f"📦 Backing up models to Google Drive: {DRIVE_MODELS_DIR}...")
for dest_name, src in artifacts.items():
    if os.path.exists(src):
        dest_path = os.path.join(DRIVE_MODELS_DIR, dest_name)
        shutil.copy2(src, dest_path)
        sz = os.path.getsize(dest_path) / 1e6
        print(f"  ✅ Drive Backup: {dest_name} ({sz:.1f} MB)")

# 2. Bundle into a single zip for instantaneous, hassle-free download
ZIP_OUT = '/content/formfix_models_26_classes.zip'
print(f"\n📦 Bundling into {ZIP_OUT}...")
with zipfile.ZipFile(ZIP_OUT, 'w', compression=zipfile.ZIP_DEFLATED) as zipf:
    for dest_name, src in artifacts.items():
        if os.path.exists(src):
            # Put models inside 'models/' subfolder so extracting straight into AI-Trainer works perfectly
            arcname = f"models/{dest_name}" if dest_name.endswith(('.keras', '.json')) else dest_name
            zipf.write(src, arcname=arcname)
            print(f"  + Added {arcname}")

drive_zip_path = os.path.join(DRIVE_DIR, 'formfix_models_26_classes.zip')
shutil.copy2(ZIP_OUT, drive_zip_path)
print(f"✅ Also saved zip copy to Google Drive: {drive_zip_path} ({os.path.getsize(ZIP_OUT)/1e6:.1f} MB)")

# 3. Trigger Automatic Browser Download
print("\n⚡ Triggering AUTOMATIC DOWNLOAD to your local computer...")
files.download(ZIP_OUT)
print("🎉 Done! Check your computer's Downloads folder for formfix_models_26_classes.zip.")
